# IAN 620 — Module 8: Large Language Models and Applications

This notebook is a practical companion to Module 8.

We will use one small local model throughout the notebook:

**Qwen/Qwen3-0.6B**

The notebook follows this path:

**text → tokens → token IDs → logits → probabilities → generation → chat → prompting → structured output → evaluation → LoRA → RAG/tool bridge**

## 1. Create a Python Environment

Before installing the LLM libraries, create a **separate Python environment** for this notebook.

A project environment keeps the Module 8 packages separate from your system Python and other projects. This reduces version conflicts and makes the notebook easier to reproduce.

- Recommended Python: **3.11**
- Python `venv` folder: **`.venv`**
- Conda environment: **`ian620-module8`**
- Choose **one** method. You do not need both `venv` and Conda.

Expand only the instructions that match your setup.

<details>
<summary><strong>Windows — Python venv</strong></summary>

<br>

Create the environment:

```powershell
py -m venv .venv
```

Activate it in PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
```

Or activate it in Command Prompt:

```bat
.venv\Scripts\activate.bat
```

</details>

<details>
<summary><strong>macOS or Linux — Python venv</strong></summary>

<br>

Create the environment:

```bash
python3 -m venv .venv
```

Activate it:

```bash
source .venv/bin/activate
```

</details>

<details>
<summary><strong>Anaconda or Miniconda</strong></summary>

<br>

Create the environment:

```bash
conda create -n ian620-module8 python=3.11 -y
```

Activate it:

```bash
conda activate ian620-module8
```

</details>

<details>
<summary><strong>VS Code — select the environment and notebook kernel</strong></summary>

<br>

1. Open the notebook folder in VS Code.
2. Open the Command Palette.
3. Select **Python: Select Interpreter**.
4. Choose `.venv` or `ian620-module8`.
5. Open this notebook.
6. Use the kernel selector in the upper-right corner.
7. Choose the same environment.

</details>

> Select the environment as the notebook kernel before installing the Module 8 libraries.


### Optional — create `.venv` automatically

Run the next cell only when working locally. It creates `.venv`, installs `ipykernel`, and registers **Python (IAN 620 Module 8)** as a notebook kernel.

Google Colab already provides an isolated environment.

> A running notebook cannot switch itself into a newly created environment. After the cell finishes, use the kernel selector and choose **Python (IAN 620 Module 8)**.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    print("Google Colab already provides an isolated environment.")
    print("You do not need to create .venv here.")

else:
    env_dir = Path(".venv")

    if env_dir.exists():
        print(".venv already exists.")
    else:
        print("Creating .venv ...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "venv",
            str(env_dir)
        ])
        print(".venv created.")

    if os.name == "nt":
        env_python = env_dir / "Scripts" / "python.exe"
    else:
        env_python = env_dir / "bin" / "python"

    subprocess.check_call([
        str(env_python),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "ipykernel"
    ])

    subprocess.check_call([
        str(env_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "ian620-module8",
        "--display-name",
        "Python (IAN 620 Module 8)"
    ])

    print()
    print("Environment setup is complete.")
    print("Now select this notebook kernel:")
    print("Python (IAN 620 Module 8)")

### Verify the Active Environment

After selecting the kernel, run the next cell and check the Python executable path.

- With `venv`, the path should normally contain `.venv`.
- With Conda, it should point to the `ian620-module8` environment.


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.system())
print("Machine:", platform.machine())


## 2. Install the libraries

In [ ]:
%pip install -q transformers==5.20.1 accelerate==1.15.0 peft==0.21.2 pandas sentence-transformers chromadb


## 3. Imports and device check

In [ ]:
import json
import time
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

## 4. Load a small local LLM

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_NAME)

# Part 1 — Tokens and Token IDs

An LLM does not read text directly. The tokenizer first converts text into tokens and then into integer token IDs.

## 5. Tokenize a sentence

In [ ]:
text = "The customer submitted a refund request."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Text:")
print(text)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

## 6. Decode token IDs back to text

In [ ]:
decoded_text = tokenizer.decode(token_ids)
print(decoded_text)

## 7. Inspect token IDs

In [ ]:
token_rows = []

for token_id in token_ids:
    token_rows.append({
        "token_id": token_id,
        "decoded_piece": tokenizer.decode([token_id])
    })

pd.DataFrame(token_rows)

# Part 2 — Logits and Next-Token Probabilities

The model produces a score for every token in its vocabulary. These raw scores are called **logits**.

## 8. Run one forward pass

In [ ]:
prompt = "The capital of France is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

print("Logits shape:", logits.shape)

## 9. View the most likely next tokens

In [ ]:
next_token_logits = logits[0, -1]

probabilities = torch.softmax(
    next_token_logits,
    dim=-1
)

top_probabilities, top_ids = torch.topk(
    probabilities,
    k=10
)

results = []

for probability, token_id in zip(top_probabilities, top_ids):
    results.append({
        "token": tokenizer.decode([token_id.item()]),
        "token_id": token_id.item(),
        "probability": float(probability)
    })

probability_df = pd.DataFrame(results)
probability_df

## 10. Visualize the probabilities

In [ ]:
probability_df.plot.bar(
    x="token",
    y="probability",
    legend=False,
    title="Top Next-Token Probabilities"
)

# Part 3 — Generation

Generation repeats the same basic process:

**predict next token → append it → predict again → append again**

## 11. Greedy generation

In [ ]:
prompt = "Machine learning is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(answer)

## 12. Compare temperature values

In [ ]:
prompt = "Write one sentence describing artificial intelligence."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

temperatures = [0.3, 0.7, 1.2]

for temperature in temperatures:
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    print(f"\nTEMPERATURE = {temperature}")
    print(text)

## 13. Compare top-p values

In [ ]:
prompt = "A useful application of language models is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

top_p_values = [0.5, 0.8, 0.95]

for top_p in top_p_values:
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    print(f"\nTOP-P = {top_p}")
    print(text)

# Part 4 — Chat Models

Chat models receive messages with roles such as `system`, `user`, and `assistant`.

## 14. Create a chat request

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a concise university teaching assistant."
    },
    {
        "role": "user",
        "content": "Explain tokenization with one simple example."
    }
]

messages

## 15. Apply the model's chat template

In [ ]:
try:
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
except TypeError:
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print(formatted_prompt)

## 16. Generate a chat response

In [ ]:
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]

response = tokenizer.batch_decode(
    new_tokens,
    skip_special_tokens=True
)[0]

print(response)

## 17. Create a reusable chat function

In [ ]:
def ask_llm(
    user_message,
    system_message="You are a concise and helpful assistant.",
    max_new_tokens=180,
    temperature=0.0
):
    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    settings = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id
    }

    if temperature > 0:
        settings.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.95
        })
    else:
        settings["do_sample"] = False

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **settings
        )

    new_tokens = output_ids[:, inputs["input_ids"].shape[1]:]

    answer = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [ ]:
answer = ask_llm(
    "What is the difference between a token and a token ID?"
)

print(answer)

# Part 5 — Prompt Engineering

A useful prompt makes the task, context, constraints, and expected output clear.

## 18. Compare a vague prompt with a clearer prompt

In [ ]:
vague_prompt = "Analyze this: The installation was easy, but the software crashes often."

clear_prompt = '''
Task:
Classify the sentiment as Positive, Negative, or Mixed.

Text:
The installation was easy, but the software crashes often.

Output:
Return only the label.
'''

print("VAGUE PROMPT")
print(ask_llm(vague_prompt))

print("\nCLEAR PROMPT")
print(ask_llm(clear_prompt))

## 19. Few-shot prompting

In [ ]:
few_shot_prompt = '''
Classify each support message as Billing, Login, or Other.

Example 1:
Message: I was charged twice.
Label: Billing

Example 2:
Message: I cannot sign in.
Label: Login

Now classify this message:
Message: My password reset link does not work.

Label:
'''

print(ask_llm(few_shot_prompt))

# Part 6 — Structured Output

Applications often need predictable data rather than free-form prose.

## 20. Ask for JSON

In [ ]:
json_prompt = '''
Read the support message and return JSON.

Message:
"I was charged twice and need a refund."

Required fields:
- category
- urgency
- summary

Return JSON only.
'''

json_response = ask_llm(
    json_prompt,
    max_new_tokens=120
)

print(json_response)

## 21. Validate the JSON

In [ ]:
try:
    parsed = json.loads(json_response)

    print("Valid JSON")
    print(parsed)

except json.JSONDecodeError:
    parsed = None
    print("The response is not valid JSON.")

## 22. Validate required fields

In [ ]:
required_fields = {
    "category",
    "urgency",
    "summary"
}

if isinstance(parsed, dict):
    missing = required_fields - set(parsed.keys())

    if missing:
        print("Missing fields:", missing)
    else:
        print("All required fields are present.")
else:
    print("No valid JSON object to validate.")

# Part 7 — Simple LLM Evaluation

A model should be tested on a fixed set of examples instead of one impressive prompt.

## 23. Create a small evaluation set

In [ ]:
evaluation_data = pd.DataFrame([
    {
        "text": "I was charged twice for my subscription.",
        "expected": "Billing"
    },
    {
        "text": "I forgot my password and cannot sign in.",
        "expected": "Login"
    },
    {
        "text": "Where can I download the user manual?",
        "expected": "Other"
    },
    {
        "text": "My invoice total looks incorrect.",
        "expected": "Billing"
    },
    {
        "text": "The reset link says it has expired.",
        "expected": "Login"
    }
])

evaluation_data

## 24. Run the same prompt across all examples

In [ ]:
predictions = []

for text in evaluation_data["text"]:
    prompt = f'''
Classify the support message as exactly one of:
Billing
Login
Other

Message:
{text}

Return only the label.
'''

    prediction = ask_llm(
        prompt,
        max_new_tokens=20
    )

    predictions.append(prediction.strip())

evaluation_data["prediction"] = predictions

evaluation_data

## 25. Calculate simple accuracy

In [ ]:
evaluation_data["correct"] = (
    evaluation_data["prediction"].str.lower()
    ==
    evaluation_data["expected"].str.lower()
)

accuracy = evaluation_data["correct"].mean()

print(f"Accuracy: {accuracy:.1%}")

evaluation_data

## 26. Measure generation time

In [ ]:
test_prompt = "Explain RAG in two short sentences."

start = time.perf_counter()

answer = ask_llm(
    test_prompt,
    max_new_tokens=100
)

elapsed = time.perf_counter() - start

print(answer)
print(f"\nGeneration time: {elapsed:.2f} seconds")

# Part 8 — LoRA Concept

Full fine-tuning updates the model's weights.

LoRA keeps the base model frozen and adds small trainable adapter matrices.

## 27. Create a LoRA configuration

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

print(lora_config)

## 28. What LoRA changes

We are not training the model in this notebook. The important point is to see how a PEFT configuration describes the adapter.

A complete training workflow would also need:

- training examples,
- tokenization,
- target modules,
- optimizer,
- trainer,
- evaluation.

# Part 9 — Chroma and the RAG Bridge

RAG does not change the model's weights.

Instead, the application retrieves useful external information and places it into the prompt. In this section, Chroma stores document vectors and retrieves the most relevant documents for a question.


## 29. What Chroma Does

Chroma is a vector database.

In this notebook, Chroma will:

1. store document text,
2. store document embeddings,
3. compare a question embedding with the stored document embeddings,
4. return the most similar documents.

The embedding model performs the semantic comparison. Chroma stores the vectors and performs the search.


## 30. Create a Small Knowledge Base

We will use a few fictional company-policy documents. Each document has an ID, title, category, and text.


In [ ]:
knowledge_base = [
    {
        "doc_id": "policy_returns",
        "title": "Standard Return Policy",
        "category": "returns",
        "text": (
            "Standard products may be returned within 14 days of delivery. "
            "The product must be in good condition and include the original packaging."
        )
    },
    {
        "doc_id": "policy_refunds",
        "title": "Refund Processing",
        "category": "refunds",
        "text": (
            "Approved refunds are sent to the original payment method. "
            "Processing normally takes five to seven business days."
        )
    },
    {
        "doc_id": "policy_shipping",
        "title": "Shipping and Delivery",
        "category": "shipping",
        "text": (
            "Standard shipping normally takes three to five business days. "
            "Tracking information becomes available after the order ships."
        )
    },
    {
        "doc_id": "policy_passwords",
        "title": "Password Reset",
        "category": "accounts",
        "text": (
            "Customers can request a password-reset link from the sign-in page. "
            "The reset link expires after 30 minutes."
        )
    },
    {
        "doc_id": "policy_warranty",
        "title": "Product Warranty",
        "category": "warranty",
        "text": (
            "Defective products may qualify for warranty service for one year "
            "from the purchase date."
        )
    }
]

knowledge_df = pd.DataFrame(knowledge_base)

knowledge_df


## 31. Create Embeddings for the Documents

We use `all-MiniLM-L6-v2` as the embedding model.

The language model and embedding model have different jobs:

- **Embedding model:** represents documents and questions as vectors for retrieval.
- **Qwen LLM:** reads the retrieved text and generates the final answer.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBEDDING_MODEL
)

document_embeddings = embedder.encode(
    knowledge_df["text"].tolist(),
    normalize_embeddings=True
)

print("Embedding matrix shape:", document_embeddings.shape)


## 32. Create a Chroma Collection

A Chroma collection stores related documents, vectors, and metadata.

This example uses an in-memory Chroma client. The collection is recreated each time this cell runs.


In [ ]:
import chromadb

chroma_client = chromadb.Client()

COLLECTION_NAME = "module8_policy_demo"

try:
    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=knowledge_df["doc_id"].tolist(),
    documents=knowledge_df["text"].tolist(),
    embeddings=document_embeddings.tolist(),
    metadatas=knowledge_df[
        ["title", "category"]
    ].to_dict(orient="records")
)

print("Documents stored in Chroma:", collection.count())


## 33. Query Chroma

We embed the question with the same embedding model and ask Chroma for the closest documents.

Because this collection uses cosine distance, a **smaller distance means greater similarity**.


In [ ]:
question = "How long do customers have to return a product?"

query_embedding = embedder.encode(
    [question],
    normalize_embeddings=True
)

query_result = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include=[
        "documents",
        "metadatas",
        "distances"
    ]
)

query_result


In [ ]:
retrieval_rows = []

for index in range(
    len(query_result["ids"][0])
):
    retrieval_rows.append({
        "rank": index + 1,
        "doc_id": query_result["ids"][0][index],
        "title": query_result["metadatas"][0][index]["title"],
        "category": query_result["metadatas"][0][index]["category"],
        "distance": round(
            float(query_result["distances"][0][index]),
            4
        ),
        "document": query_result["documents"][0][index]
    })

retrieved_df = pd.DataFrame(
    retrieval_rows
)

retrieved_df


## 34. Use Chroma Results in a RAG Prompt

The retrieval result becomes the context supplied to the LLM.

The LLM does not directly communicate with Chroma. The application:

1. searches Chroma,
2. receives relevant documents,
3. formats those documents into a prompt,
4. sends the prompt to the LLM.


In [ ]:
def retrieve_with_chroma(
    question,
    k=3
):
    query_embedding = embedder.encode(
        [question],
        normalize_embeddings=True
    )

    result = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    rows = []

    for index in range(
        len(result["ids"][0])
    ):
        rows.append({
            "rank": index + 1,
            "doc_id": result["ids"][0][index],
            "title": result["metadatas"][0][index]["title"],
            "category": result["metadatas"][0][index]["category"],
            "distance": float(
                result["distances"][0][index]
            ),
            "document": result["documents"][0][index]
        })

    return pd.DataFrame(rows)


In [ ]:
def build_chroma_rag_prompt(
    question,
    retrieved_documents
):
    source_blocks = []

    for _, row in retrieved_documents.iterrows():
        source_blocks.append(
            f"[{row['doc_id']}]\n"
            f"Title: {row['title']}\n"
            f"Content: {row['document']}"
        )

    context = "\n\n".join(
        source_blocks
    )

    return f"""
Answer the question using only the retrieved context.

Rules:
- Do not add information that is not in the context.
- Cite the supporting document ID.
- If the context does not contain the answer, say:
  "The retrieved documents do not contain enough information."

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

ANSWER:
""".strip()


In [ ]:
chroma_question = "How long do customers have to return a product?"

chroma_results = retrieve_with_chroma(
    chroma_question,
    k=3
)

chroma_prompt = build_chroma_rag_prompt(
    chroma_question,
    chroma_results
)

print("RETRIEVED DOCUMENTS")

display(
    chroma_results[
        [
            "rank",
            "doc_id",
            "title",
            "distance"
        ]
    ]
)

print("\nRAG PROMPT")
print(chroma_prompt)


In [ ]:
chroma_answer = ask_llm(
    chroma_prompt,
    system_message=(
        "You are a grounded question-answering assistant. "
        "Use only the retrieved context."
    ),
    max_new_tokens=120
)

print("RAG ANSWER")
print(chroma_answer)


### Optional — Keep the Chroma Database on Disk

The example above uses an in-memory database, so its data disappears when the Python process ends.

For a reusable local database, use:

```python
persistent_client = chromadb.PersistentClient(
    path="module8_chroma_db"
)
```

Then create or retrieve the collection from `persistent_client` instead of `chroma_client`.


## 35. Compare an ungrounded prompt and a grounded prompt

In [ ]:
question = "What is the refund deadline?"

ungrounded_prompt = question

grounded_prompt = '''
Answer using only the context below.

Context:
Refund requests must be submitted within 14 days of delivery.

Question:
What is the refund deadline?

Answer:
'''

print("WITHOUT EXTERNAL CONTEXT")
print(ask_llm(ungrounded_prompt))

print("\nWITH RETRIEVED CONTEXT")
print(ask_llm(grounded_prompt))

## 36. Build a RAG-style prompt from retrieved documents

In [ ]:
retrieved_documents = [
    {
        "source_id": "S1",
        "text": "Refund requests must be submitted within 14 days of delivery."
    },
    {
        "source_id": "S2",
        "text": "Customers can submit refund requests from the Orders page."
    }
]

context = "\n\n".join(
    f"[{doc['source_id']}] {doc['text']}"
    for doc in retrieved_documents
)

rag_prompt = f'''
Use only the context below.

Context:
{context}

Question:
{question}

Requirements:
- Give a concise answer.
- Cite the supporting source IDs.
'''

print(rag_prompt)

In [ ]:
rag_answer = ask_llm(
    rag_prompt,
    max_new_tokens=100
)

print(rag_answer)

# Part 10 — Bridge to Tool Calling and Agents

A model can also produce structured data that an application interprets as a request to call a function.

## 37. Ask the model to choose a tool

In [ ]:
tool_prompt = '''
You may choose one of these tools:

1. search_policy
   Use when the user asks about company rules or policies.

2. lookup_order
   Use when the user asks about a specific order.

User request:
"What is the return policy?"

Return JSON only:
{
  "tool": "...",
  "reason": "..."
}
'''

tool_response = ask_llm(
    tool_prompt,
    max_new_tokens=100
)

print(tool_response)

## 38. Validate before executing anything

In [ ]:
allowed_tools = {
    "search_policy",
    "lookup_order"
}

try:
    tool_request = json.loads(tool_response)

    selected_tool = tool_request.get("tool")

    if selected_tool in allowed_tools:
        print("Approved tool:", selected_tool)
    else:
        print("Rejected tool request:", selected_tool)

except json.JSONDecodeError:
    print("Tool request was not valid JSON.")

# Module 8 Summary

You have now worked through the main Module 8 ideas in code:

1. Tokenization and token IDs
2. Logits and probabilities
3. Greedy generation
4. Temperature and top-p
5. Chat messages and chat templates
6. Local LLM inference
7. Prompt engineering
8. Structured JSON output
9. Simple evaluation
10. LoRA configuration
11. Chroma vector storage and semantic retrieval
12. RAG grounding with retrieved documents
13. Tool-calling validation

The next step is **Module 9 — Agentic RAG**, where retrieval and tools can be used inside a controlled decision loop.